In [5]:
#PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed"

#!wget {PREFIX}/download.py
#!wget {PREFIX}/embedder.py

--2026-06-29 17:09:05--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py.1’

download.py.1       100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-06-29 17:09:06 (83.2 MB/s) - ‘download.py.1’ saved [1376/1376]

--2026-06-29 17:09:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 

In [9]:
!python download.py

  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [10]:
from embedder import Embedder

In [12]:
embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"
v1 = embed.encode(q1)
v1[0]

np.float64(-0.02058203437252893)

In [32]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]


In [35]:
texts = []

for doc in documents:
    text = doc["content"] + " " + doc["filename"]
    texts.append(text)

In [39]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

  0%|          | 0/2 [00:00<?, ?it/s]

In [40]:
import numpy as np
X = np.array(vectors)

In [44]:
scores = X.dot(v1)
scores

array([ 0.31518771, -0.07676619, -0.02814066,  0.08876018,  0.55103467,
        0.0763816 , -0.03591069,  0.07590576,  0.25770909,  0.21671927,
        0.20521284,  0.10541617,  0.04946655,  0.22375971,  0.05297868,
        0.04281173,  0.29250964,  0.25866384,  0.05960215,  0.40618203,
        0.35449249,  0.23261767,  0.36107027,  0.28324618, -0.01112025,
        0.17957483,  0.04493099,  0.11867115, -0.04785281,  0.08226051,
       -0.01681499,  0.12729314,  0.0410838 , -0.03772213,  0.03382858,
        0.34872972,  0.33683696,  0.06261032,  0.32395558,  0.39668211,
        0.28161163,  0.28598972,  0.23595931,  0.05080852,  0.06672208,
        0.18874446,  0.19559025,  0.00644912,  0.00827642,  0.09649605,
        0.05421206,  0.08674141,  0.01312468,  0.04199086,  0.16987396,
        0.1099587 ,  0.02092688, -0.00990826, -0.10578756,  0.00479905,
        0.24731703,  0.34550636,  0.40610592,  0.15960919,  0.33647357,
       -0.03078465,  0.27183543,  0.1769876 , -0.06072496, -0.01

In [55]:
filenames = [doc["filename"] for doc in documents]

target = "02-vector-search/lessons/07-sqlitesearch-vector.md"
idx = filenames.index(target)

similarity = v1.dot(X[idx])

print(idx)
print(filenames[idx])
print(similarity)


22
02-vector-search/lessons/07-sqlitesearch-vector.md
0.36107026789538205


In [56]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [59]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np

texts = [chunk["content"] for chunk in chunks]

vectors = []

for i in tqdm(range(0, len(texts), 50)):
    batch = texts[i:i + 50]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

scores = X.dot(v1)

idx = np.argmax(scores)

print(scores[idx])
print(chunks[idx]["filename"])

  0%|          | 0/6 [00:00<?, ?it/s]

0.6489016436447387
02-vector-search/lessons/07-sqlitesearch-vector.md


In [63]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [64]:
from minsearch import Index

def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )

    index.fit(documents)

    return index

In [65]:
query = "How do I store vectors in PostgreSQL?"

In [66]:
text_index = build_index(chunks)

text_results = text_index.search(
    query,
    num_results=5
)

text_files = [r["filename"] for r in text_results]

text_files

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [67]:
texts = [chunk["content"] for chunk in chunks]

vectors = []

for i in tqdm(range(0, len(texts), 50)):
    batch = texts[i:i + 50]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

In [68]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

query_vector = embed.encode(query)

vector_results = vindex.search(
    query_vector,
    num_results=5
)

vector_files = [r["filename"] for r in vector_results]

vector_files

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [69]:
set(vector_files) - set(text_files)

{'02-vector-search/lessons/08-pgvector.md'}

In [70]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [71]:
query = "How do I give the model access to tools?"

In [72]:
text_results = text_index.search(
    query,
    num_results=5
)

In [73]:
query_vector = embed.encode(query)

vector_results = vindex.search(
    query_vector,
    num_results=5
)

In [75]:
results = rrf([vector_results, text_results])


In [76]:
results[0]["filename"], results[0]["start"]

('01-agentic-rag/lessons/13-function-calling.md', 4000)